# Notebook 1: Introduction to DeepSeek V3.2 Architecture

## Learning Objectives
- Understand the overall architecture of DeepSeek V3.2
- Learn about Mixture of Experts (MoE) fundamentals
- Understand why sparse activation matters
- Get familiar with key innovations: MLA, DSA, MTP

## Task 1: DeepSeek V3.2 Overview

DeepSeek V3.2 is a state-of-the-art 671B parameter Mixture-of-Experts (MoE) language model with:
- **671B total parameters** but only **37B activated** per token
- **128K context window**
- Key innovations: DeepSeek Sparse Attention (DSA), Multi-Head Latent Attention (MLA), Multi-Token Prediction (MTP)

## Task 2: Why Mixture of Experts?

### HINT: Traditional Dense vs Sparse MoE
- **Dense models**: All parameters activated for every token → computationally expensive
- **MoE**: Only a small subset of "experts" activated per token → efficiency
- DeepSeek V3.2 has **256 experts** but only activates **8-9 per token**

In [ ]:
# Exercise: Compare dense vs MoE parameter efficiency
# TODO: Calculate the sparsity ratio for DeepSeek V3.2
# Total params: 671B, Active params: 37B

total_params = 671e9  # 671 billion
active_params = 37e9   # 37 billion

sparsity_ratio = 1 - (active_params / total_params)
print(f"Sparsity ratio: {sparsity_ratio:.2%}")
print(f"Parameter efficiency: {active_params/total_params:.2%}")

## Task 3: Key Architectural Components

### HINT: Three Major Innovations
1. **Multi-Head Latent Attention (MLA)**: Compresses KV cache into lower-dimensional latent space
2. **DeepSeek Sparse Attention (DSA)**: Reduces O(L²) to O(L·k) complexity
3. **Multi-Token Prediction (MTP)**: Predicts multiple tokens ahead during training

In [ ]:
# Exercise: Visualize the architecture components
import matplotlib.pyplot as plt

components = {
    'MLA': 'KV Compression',
    'MoE': 'Sparse Expert Routing',
    'DSA': 'Sparse Attention',
    'MTP': 'Multi-Token Prediction',
    'RoPE': 'Rotary Position Embedding'
}

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(list(components.keys()), [1]*len(components), color='steelblue')
ax.set_xlabel('Component Importance')
ax.set_title('DeepSeek V3.2 Key Components')
plt.tight_layout()
plt.show()

print("\nComponent descriptions:")
for comp, desc in components.items():
    print(f"  {comp}: {desc}")

## Task 4: Simplified Model Specification

### HINT: Our Implementation Scale
For learning purposes, we'll implement a simplified version:
| Component | Full DeepSeek V3.2 | Our Implementation |
|-----------|-------------------|-------------------|
| Hidden Dim | 7168 | 512 |
| Layers | 61 | 4 |
| Experts | 256 | 8 |
| Active Experts | 8-9 | 2 |
| Attention Heads | 128 | 8 |
| Context | 128K | 512 |

In [ ]:
# Exercise: Define our simplified model config
class DeepSeekV3Config:
    """Simplified DeepSeek V3.2 configuration for learning"""
    def __init__(self):
        self.vocab_size = 50000
        self.hidden_size = 512      # vs 7168 in full
        self.intermediate_size = 1376  # ~2.5x hidden
        self.num_hidden_layers = 4    # vs 61 in full
        self.num_attention_heads = 8   # vs 128 in full
        self.num_kv_heads = 1          # GQA
        self.num_experts = 8          # vs 256 in full
        self.num_active_experts = 2   # vs 8-9 in full
        self.max_position_embeddings = 512  # vs 128K
        self.rms_norm_eps = 1e-6
        self.rope_theta = 10000.0
        
    def __repr__(self):
        return f"DeepSeekV3Config(hidden_size={self.hidden_size}, layers={self.num_hidden_layers})"

config = DeepSeekV3Config()
print(config)

# Calculate total parameters (simplified)
def estimate_params(config):
    """Estimate model parameters"""
    # Embedding params
    embed_params = config.vocab_size * config.hidden_size * 2
    
    # Attention params per layer
    attn_params = config.hidden_size * config.hidden_size * 3  # Q, K, V projections
    out_params = config.hidden_size * config.hidden_size
    
    # MoE params per layer
    expert_params = config.intermediate_size * config.hidden_size * config.num_experts
    gate_params = config.hidden_size * config.num_experts
    
    # Total per layer
    per_layer = attn_params + out_params + expert_params + gate_params
    
    total = embed_params + (per_layer * config.num_hidden_layers)
    return total

print(f"\nEstimated parameters: {estimate_params(config)/1e6:.1f}M")

## Verification

Complete these checks:
1. ✅ Can explain why MoE is efficient
2. ✅ Can list the three key innovations (MLA, DSA, MTP)
3. ✅ Understand the difference between total and active parameters
4. ✅ Know our simplified model specifications